In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import matplotlib.pyplot as plt
import seaborn as sns


In [12]:
df = pd.read_csv("../data/fake_jobs_cleaned.csv")
df.head()


,title,location,description,requirements,telecommuting,has_company_logo,has_questions,fraudulent,text,clean_text
0,Architect (Middleware - MQ) - Kuwait,"KW, KU,","On behalf of our client, a well known multinat...",-Working technical knowledge of IT systems and...,0,1,0,0,Architect (Middleware - MQ) - Kuwait On behalf...,architect middleware mq kuwait on behalf of ou...
1,Interviewing Now for Sales Rep Positions -- wi...,"US, TX, Corpus Christi","We are Argenta Field Solutions, a rapidly expa...",#NAME?,0,1,0,0,Interviewing Now for Sales Rep Positions -- wi...,interviewing now for sales rep positions with ...
2,Process Controls Staff Engineer - Foxboro I/A ...,"US, TX, USA Southwest",Experienced Process Controls Staff Engineer is...,At least 10 years of degreed professional expe...,0,0,0,0,Process Controls Staff Engineer - Foxboro I/A ...,process controls staff engineer foxboro ia tri...
3,Experienced Telemarketer Wanted - Digital Solu...,"AU, NSW,",If you have a passion for people and love to s...,"Responsibilities - Prospecting, following up a...",0,1,0,0,Experienced Telemarketer Wanted - Digital Solu...,experienced telemarketer wanted digital soluti...
4,Senior Network Engineer,"GB, ENG, London",As the successful Senior Network Engineer you ...,Essential skills:•Juniper switching/routing/se...,0,1,0,0,Senior Network Engineer As the successful Seni...,senior network engineer as the successful seni...


In [13]:
X = df['clean_text']
y = df['fraudulent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((7068,), (1767,))

In [14]:
%pip install transformers sentencepiece torch --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

In [16]:
def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=256)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].numpy()

In [17]:
X_train_bert = np.vstack([get_bert_embedding(t) for t in X_train])
X_test_bert = np.vstack([get_bert_embedding(t) for t in X_test])

X_train_bert.shape, X_test_bert.shape

((7068, 384), (1767, 384))

In [18]:
np.save("../models/X_train_bert.npy", X_train_bert)
np.save("../models/X_test_bert.npy", X_test_bert)

print("BERT features saved!")

BERT features saved!
